# LangGraph Core: State, Nodes & Edges

LangGraph models your application as a **directed graph** where:
- **State** is a `TypedDict` shared across all nodes
- **Nodes** are Python functions that read from state and return partial updates
- **Edges** define execution order (which node runs next)

Topics:
- `StateGraph`, `START`, `END`
- Simple state with overwrite semantics
- **Accumulating state** with `Annotated[list, operator.add]`
- **`add_messages` reducer** for chat state
- Multi-node sequential pipeline

In [ ]:
import operator
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END, add_messages
from typing_extensions import TypedDict, Annotated

## 1. The Simplest Graph

A `StateGraph` is built in 4 steps:
1. Define the `State` TypedDict
2. Define node functions (`state → dict`)
3. Add nodes and edges
4. Compile → executable `app`

In [ ]:
class SimpleState(TypedDict):
    input: str
    output: str
    step: int

def process(state: SimpleState) -> dict:
    # Nodes return a dict — only the keys returned are updated in state
    return {"output": state["input"].upper(), "step": state["step"] + 1}

graph = StateGraph(SimpleState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)

app = graph.compile()
result = app.invoke({"input": "hello", "output": "", "step": 0})

print(f"Input: {result['input']}")
print(f"Output: {result['output']}")
print(f"Step: {result['step']}")

## 2. Visualize the Graph

LangGraph can render a Mermaid diagram of any compiled graph.

In [ ]:
print(app.get_graph().draw_mermaid())

## 3. State Reducers — Accumulating Values

By default, node return values **overwrite** state keys. Use `Annotated[T, reducer]` to change this:

| Annotation | Behavior |
|---|---|
| `Annotated[list[str], operator.add]` | Lists are **concatenated** |
| `Annotated[int, operator.add]` | Integers are **summed** |
| `Annotated[list[BaseMessage], add_messages]` | Messages are **appended** (LangGraph built-in) |

This is critical for parallel branches that all write to the same key.

In [ ]:
class AccumulatingState(TypedDict):
    messages: Annotated[list[str], operator.add]  # lists concatenate
    count:    Annotated[int,       operator.add]  # integers sum

def step_one(state: AccumulatingState) -> dict:
    return {"messages": ["Step 1 executed"], "count": 1}

def step_two(state: AccumulatingState) -> dict:
    return {"messages": ["Step 2 executed"], "count": 1}

g = StateGraph(AccumulatingState)
g.add_node("step_one", step_one)
g.add_node("step_two", step_two)
g.add_edge(START, "step_one")
g.add_edge("step_one", "step_two")
g.add_edge("step_two", END)

result = g.compile().invoke({"messages": ["Initial"], "count": 0})

print(f"Messages: {result['messages']}")
print(f"Count: {result['count']}")

## 4. Message State with `add_messages`

`add_messages` is LangGraph's built-in reducer for chat messages. It handles deduplication (by `id`) and works seamlessly with LangChain message types.

This pattern is the foundation of all LangGraph chatbots.

In [ ]:
class MessageState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = init_chat_model("gpt-4o-mini", temperature=0)

def chat_node(state: MessageState) -> dict:
    # LLM receives all messages; its response is appended
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

g = StateGraph(MessageState)
g.add_node("chat", chat_node)
g.add_edge(START, "chat")
g.add_edge("chat", END)

app = g.compile()
result = app.invoke({"messages": [HumanMessage(content="Say Hello in Tagalog")]})

for msg in result["messages"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"{role}: {msg.content}")

## 5. Multi-Node Sequential Pipeline

Nodes run in a defined sequence. Each node's output is available to the next. This enables **agentic pipelines** where each node enriches the state.

In [ ]:
class MultiStepState(TypedDict):
    input: str
    analyzed: str
    enhanced: str
    final: str

def analyze_node(state: MultiStepState) -> dict:
    resp = llm.invoke(f"Analyze and summarize in one sentence: {state['input']}")
    return {"analyzed": resp.content}

def enhance_node(state: MultiStepState) -> dict:
    resp = llm.invoke(f"Enhance with more detail: {state['analyzed']}")
    return {"enhanced": resp.content}

def finalize_node(state: MultiStepState) -> dict:
    resp = llm.invoke(f"Write a concise final summary: {state['enhanced']}")
    return {"final": resp.content}

g = StateGraph(MultiStepState)
g.add_node("analyze",  analyze_node)
g.add_node("enhance",  enhance_node)
g.add_node("finalize", finalize_node)

g.add_edge(START, "analyze")
g.add_edge("analyze", "enhance")
g.add_edge("enhance", "finalize")
g.add_edge("finalize", END)

result = g.compile().invoke({"input": "Artificial intelligence", "analyzed": "", "enhanced": "", "final": ""})

print(f"Input:    {result['input']}")
print(f"Analyzed: {result['analyzed'][:100]}...")
print(f"Enhanced: {result['enhanced'][:100]}...")
print(f"Final:    {result['final']}")

## 6. Exercise: Question Generator + Answerer

Build a 2-node graph:
1. **Node 1**: Generate 3 questions about a topic
2. **Node 2**: Answer the first question

Return both questions and the answer.

In [ ]:
class QAState(TypedDict):
    topic: str
    questions: str
    answer: str

def generate_questions(state: QAState) -> dict:
    resp = llm.invoke(f"Generate 3 interesting questions about: {state['topic']}\nFormat: numbered list")
    return {"questions": resp.content}

def answer_question(state: QAState) -> dict:
    resp = llm.invoke(f"Answer the first question from this list:\n{state['questions']}")
    return {"answer": resp.content}

g = StateGraph(QAState)
g.add_node("generate", generate_questions)
g.add_node("answer",   answer_question)
g.add_edge(START, "generate")
g.add_edge("generate", "answer")
g.add_edge("answer", END)

result = g.compile().invoke({"topic": "renewable energy", "questions": "", "answer": ""})
print(f"Topic: {result['topic']}")
print(f"Questions:\n{result['questions']}")
print(f"\nAnswer:\n{result['answer']}")